# Lab - Human in the Loop: A Review Queue with an SLA

Three tasks that put a person between ShopAssist and a large refund - the rule in the tool, the decision in the code, the explanation from the model - and treat the queue itself as part of the system, with a promise to keep.

| Task | What you learn |
| --- | --- |
| 1 | Refunds above the limit go to a review queue on disk, not through |
| 2 | The person decides, the code refunds, Claude explains - with no tools |
| 3 | A rejected case, a late case, and a report against the 24-hour SLA |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json
import time
from pathlib import Path

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
LIMIT = 100                    # the business's rule: refunds above this need a human
QUEUE = "review_queue.json"
Path(QUEUE).write_text("[]")   # an empty queue for this session

SYSTEM = ("You are ShopAssist. Use process_refund for refunds. "
          "Answer the customer in two sentences.")
FIELDS = {"order_id": "string", "amount": "number", "reason": "string"}
TOOLS = [{
    "name": "process_refund",
    "description": "Refund an amount for an order.",
    "input_schema": {"type": "object", "required": list(FIELDS),
                     "properties": {f: {"type": t} for f, t in FIELDS.items()}},
}]


def load_queue():
    return json.loads(Path(QUEUE).read_text())


def save_queue(queue):
    Path(QUEUE).write_text(json.dumps(queue, indent=2))


def agent(text, process_refund):
    """The familiar loop, with process_refund as its only tool."""
    messages = [{"role": "user", "content": text}]
    results = []
    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=300, output_config={"effort": "low"},
            system=SYSTEM, tools=TOOLS, messages=messages)
        messages.append({"role": "assistant", "content": response.content})
        if response.stop_reason != "tool_use":
            return results, " ".join(b.text for b in response.content if b.type == "text")
        outputs = []
        for block in response.content:
            if block.type == "tool_use":
                out = process_refund(**block.input)
                results.append(out)
                print(f"  process_refund {block.input} -> {out}")
                outputs.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": json.dumps(out)})
        messages.append({"role": "user", "content": outputs})

---

## Task 1 - The rule lives in the tool

In [ ]:
# ============================================================
# TASK 1 - The rule lives in the tool
# ============================================================
#
# WHAT TO DO
#   Write process_refund(): up to the limit it refunds; above
#   it, it adds a case to the review queue - a ticket, the order,
#   the amount, the reason, when it arrived, and an empty
#   decision - and returns pending review. Send two requests: 60
#   for a jacket, 250 for a broken TV.
#
# WHY IT MATTERS
#   Some decisions should not be made by a model at all, and the
#   business has said so. Put the human step in the prompt and
#   the rule depends on the model remembering it in every
#   conversation. In the tool, it holds every time - and the
#   tool never asks Claude whether to make an exception.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Note what it does not do: it never asks Claude whether to
#   make an exception".
#
# HOW TO DO IT
#   Three blanks:
#
#       LIMIT                        the automatic limit
#       f"REV-{len(queue) + 1}"      the next ticket number
#       "pending_review"             what the customer is told
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def process_refund(order_id, amount, reason):
        if amount <= ...:   # LIMIT
            return {"status": "refunded", "refund_id": "REF-7781"}
        queue = load_queue()
        ticket = ...   # f"REV-{len(queue) + 1}"
        queue.append({"ticket": ticket, "order_id": order_id, "amount": amount,
                      "reason": reason, "created_at": time.time(), "decision": None})
        save_queue(queue)
        return {"status": ..., "ticket": ticket,   # "pending_review"
                "customerMessage": "A person will review it."}

    runs = {}
    for text in ["Refund 60 for the jacket in ORD-12345678, it was too small.",
                 "Refund 250 for the broken TV in ORD-87654321."]:
        print("Customer:", text)
        runs[text] = agent(text, process_refund)
        print("ShopAssist:", runs[text][1])
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print("queue:", load_queue())

check("rq_tool", runs=runs, queue=load_queue())

---

## Task 2 - The person decides, the code acts, Claude explains

In [ ]:
# ============================================================
# TASK 2 - The person decides, the code acts, Claude explains
# ============================================================
#
# WHAT TO DO
#   Write review(): find the case, record the decision and who
#   made it, and - if approved - let the code carry out the
#   refund and store its ID. Then Claude writes the update to the
#   customer from the reviewed case, in a call with no tools at
#   all. Approve REV-1.
#
# WHY IT MATTERS
#   The human decides and the code acts; the model only talks.
#   With no tools in the update call, Claude cannot refund twice
#   or act on its own reading of the case. And the queue on disk
#   - not the conversation - holds the decision, who made it and
#   when: the audit trail.
#
# WHERE TO SEE IT IN THE LECTURE
#   "This call has no tools at all, so Claude cannot refund
#   twice".
#
# HOW TO DO IT
#   Three blanks:
#
#       "ticket"                     the case being decided
#       "approved"                   only then does money move
#       json.dumps(case)             the reviewed case, for the update
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def review(ticket, decision, reviewer):
        queue = load_queue()
        case = next(c for c in queue if c[...] == ticket)   # "ticket"
        case.update({"decision": decision, "reviewer": reviewer, "decided_at": time.time()})
        if decision == ...:   # "approved"
            case["refund_id"] = f"REF-{7789 + int(ticket.split('-')[1])}"   # code refunds, not Claude
        save_queue(queue)
        request = {"model": MODEL, "max_tokens": 200, "output_config": {"effort": "low"},
                   "system": SYSTEM,
                   "messages": [{"role": "user", "content":
                                 "Write the customer update for this reviewed refund: "
                                 + ...}]}   # json.dumps(case)
        response = client.messages.create(**request)
        return " ".join(b.text for b in response.content if b.type == "text"), request

    update, update_request = review("REV-1", "approved", "dana")
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Task 1 included."
                     .format(exc)) from None
except StopIteration:   # provided: the queue holds no such case yet
    raise ValueError("The queue holds no case REV-1 yet - finish Task 1 first.") from None
print("ShopAssist:", update)
print("queue:", load_queue())

check("rq_review", queue=load_queue(), update=update, request=update_request)

---

## Task 3 - The queue has an SLA too

In [ ]:
# ============================================================
# TASK 3 - The queue has an SLA too
# ============================================================
#
# WHAT TO DO
#   A third request lands in the queue and is rejected; one case
#   from yesterday is still waiting. Write queue_report(): what
#   is pending, which pending cases are past the 24-hour SLA,
#   and the share of decided cases that were approved.
#
# WHY IT MATTERS
#   A review queue moves the risk from the model to people - and
#   people have capacity. An architect designs the queue as part
#   of the system: a promise to the customer ("a person will
#   reply within a day"), a report on what is late, and an
#   approval rate that tells whether the limit is set well.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Keep pending cases outside the conversation, let the code
#   carry out the reviewer's decision".
#
# HOW TO DO IT
#   Three blanks:
#
#       "decision"                           None: still waiting
#       "created_at"                         older than the SLA: late
#       approved / len(decided)              the approval rate
# ============================================================

SLA_HOURS = 24

try:
    results, answer = agent("Refund 180 for the scratched desk in ORD-55554444.", process_refund)
    print("ShopAssist:", answer)
    rejected_update, _ = review(results[0]["ticket"], "rejected", "sam")
    print("ShopAssist:", rejected_update)
    queue = load_queue()   # provided: a case that arrived yesterday morning, still waiting
    queue.append({"ticket": f"REV-{len(queue) + 1}", "order_id": "ORD-24681357", "amount": 140,
                  "reason": "parcel lost", "created_at": time.time() - 30 * 3600, "decision": None})
    save_queue(queue)

    # TODO: replace each ... below with the value named beside it
    def queue_report(queue, now):
        pending = [c for c in queue if c[...] is None]   # "decision"
        decided = [c for c in queue if c["decision"] is not None]
        approved = sum(c["decision"] == "approved" for c in decided)
        return {"pending": [c["ticket"] for c in pending],
                "over_sla": [c["ticket"] for c in pending
                             if now - c[...] > SLA_HOURS * 3600],   # "created_at"
                "approval_rate": ... if decided else None}   # approved / len(decided)

    report = queue_report(load_queue(), time.time())
    if ... in report.values():   # provided: a blank left above, said plainly
        raise TypeError("ellipsis")
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Tasks 1 to 2 included."
                     .format(exc)) from None
for key, value in report.items():
    print(f"{key:14} {value}")

check("rq_sla", report=report, queue=load_queue(), rejected_update=rejected_update)

---

## Done

Put human approval inside the tool that takes the action, keep pending cases outside the conversation, let the code carry out the reviewer's decision and the model only explain it - and run the queue against an SLA.